# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression.

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept).
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already.
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [1]:
NAMES = "Erik H., Derek W., Henry Y."

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

For a binary MLP, a Binary Cross-Entropy Cost Function is needed to perform it. For a multi-class MLP, a Categorical Cross-Entropy is needed instead. Meanwhile, linear regression MLPs need a MSE function as its cost function.

For both binary and linear regression MLPs, one output neuron is needed, whereas the number of neurons needed for a multiclass MLP depends on the number of classes. In terms of the activation function, a binary MLP requires a Sigmoid function, a multiclass MLP requires a Softmax function, and a linear regression MLP doesn't even require a specific activation function.

When we standardize a function, each of the features will have an equal level of input into the function, so it prevents functions with disparagingly high values from dominating the learning process of the model.

In order to prevent overfitting, if the dataset isn't too large, cross-validation can be used as a remedy, as it exposes the model to unseen data. If the data adjusts well to it, then it should be fine. Otherwise, there may be some overfitting involved.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import torch
torch.set_num_threads(1)

print("PyTorch threads:", torch.get_num_threads())

PyTorch threads: 1


In [2]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [3]:
# USE AS MANY CELLS AS NEEDED

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [3]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3,
                           n_clusters_per_class=1,
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
import torch


class MultiClassMLP:

    def __init__(self, input_dim, hidden_dim, n_classes, hidden_dim2=None, learning_rate=0.01, activation="relu"):
        self.hidden_dim2 = hidden_dim2
        self.activation = activation
        self.learning_rate = learning_rate

        self.W_1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
        self.b_1 = torch.zeros(hidden_dim, requires_grad=True)

        if hidden_dim2 is None:
            self.W_out = torch.randn(
                hidden_dim, n_classes, requires_grad=True
            )
        else:
            self.W_2 = torch.randn(
                hidden_dim, hidden_dim2, requires_grad=True
            )
            self.b_2 = torch.zeros(hidden_dim2, requires_grad=True)
            self.W_out = torch.randn(
                hidden_dim2, n_classes, requires_grad=True
            )

        self.b_out = torch.zeros(n_classes, requires_grad=True)

    def activate(self, z):
        if self.activation == "relu":
            return torch.relu(z)
        else:
            return torch.tanh(z)

    def forward(self, X):
        h = self.activate(X @ self.W_1 + self.b_1)

        if self.hidden_dim2 is not None:
            h = self.activate(h @ self.W_2 + self.b_2)

        out = h @ self.W_out + self.b_out
        return torch.softmax(out, dim=1)

    def cost(self, y_pred, y_true):
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)

        return -torch.mean(
            torch.sum(y_true * torch.log(y_pred), dim=1)
        )

    def fit(self, X, y, n_epochs=100):
        for epoch in range(n_epochs):
            randperm = torch.randperm(X.shape[0])

            for i in randperm:
                x_batch, y_batch = X[i:i + 1], y[i:i + 1]

                # forward pass
                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)

                # backward pass
                cost.backward()

                # update parameters 
                with torch.no_grad():
                    self.W_1 -= self.learning_rate * self.W_1.grad
                    self.b_1 -= self.learning_rate * self.b_1.grad
                    self.W_out -= self.learning_rate * self.W_out.grad
                    self.b_out -= self.learning_rate * self.b_out.grad

                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_out.grad.zero_()
                    self.b_out.grad.zero_()

                    if self.hidden_dim2 is not None:
                        self.W_2 -= self.learning_rate * self.W_2.grad
                        self.b_2 -= self.learning_rate * self.b_2.grad

                        self.W_2.grad.zero_()
                        self.b_2.grad.zero_()

        return self

    def predict(self, X):
        with torch.no_grad():
            probs = self.forward(X)
            predicted = torch.argmax(probs, dim=1)

        return predicted

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# split into training and holdout sets (80/20)
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

# one-hot encoder
encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = encoder.fit_transform(y_train.reshape(-1, 1))

# convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train_onehot = torch.tensor(y_train_onehot, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_holdout = torch.tensor(y_holdout)

print("X_train:", X_train.shape, " y_train_onehot:", y_train_onehot.shape)
print("X_holdout:", X_holdout.shape, " y_holdout:", y_holdout.shape)

X_train: torch.Size([800, 2])  y_train_onehot: torch.Size([800, 3])
X_holdout: torch.Size([200, 2])  y_holdout: torch.Size([200])


In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score

torch.manual_seed(1234)

learning_rate = 0.01
epochs = 150
results = []

for activation in ["relu", "tanh"]:
    for n_layers in [1, 2]:
        for units in [8, 16, 32]:
            if n_layers == 1:
                hidden_dim2 = None
            else:
                hidden_dim2 = units

            # create and train a fresh model for each configuration
            model = MultiClassMLP(input_dim=2, hidden_dim=units, n_classes=3, hidden_dim2=hidden_dim2, learning_rate=learning_rate, activation=activation)
            model.fit(X_train, y_train_onehot, n_epochs=epochs)

            # accuracy on both the training and holdout sets
            results.append({
                "Activation": activation,
                "Hidden layers": n_layers,
                "Units per layer": units,
                "Train accuracy": accuracy_score(y_train, model.predict(X_train)),
                "Holdout accuracy": accuracy_score(y_holdout, model.predict(X_holdout))
            })

results_df = pd.DataFrame(results)
results_df

,Activation,Hidden layers,Units per layer,Train accuracy,Holdout accuracy
0,relu,1,8,0.82250,0.805
1,relu,1,16,0.81625,0.800
2,relu,1,32,0.82375,0.790
3,relu,2,8,0.81875,0.800
4,relu,2,16,0.82250,0.800
5,relu,2,32,0.83500,0.825
6,tanh,1,8,0.82750,0.795
7,tanh,1,16,0.81750,0.800
8,tanh,1,32,0.81875,0.805
9,tanh,2,8,0.82875,0.805


### Reflection and discussion (multi-class)

The 12 configurations all got similar results, about 81.6–84.3% accuracy on the training set and 79–82.5% on the holdout set. This is close to the best we can expect for this dataset because flip_y=0.15 randomly reassigns 15% of the labels, so roughly 10% of the points have the wrong label, and no model can predict them.

Different Hidden Units: With one hidden layer, going from 8 to 16 to 32 units changes holdout accuracy by at most 1.5 percentage points (79–80.5%). Fairly simple boundaries separate the classes, so 8 units already give the model enough capacity, and adding more units does not help.

ReLU vs. Tanh: The activation function creates a little difference. ReLU models score 79–82.5% on the holdout set and Tanh models 79.5–82%. Tanh is slightly more consistent, likely because its output is smooth and centered at zero, while ReLU creates piecewise-linear boundaries. For this data, both activations are flexible enough, and the label noise limits performance more than the choice of activation.

Different Hidden Layer: Adding a second layer slightly increases the highest training accuracy, from 82.8% to 84.3%, and yields a few of the best holdout scores (82–82.5%). However, the improvement is small and not consistent (for example, ReLU with 8 or 16 units does no better than one layer). A second layer adds flexibility. However, on a simple, noisy dataset like this, it mostly fits the training data a bit better rather than generalizing better. The gap between training and holdout accuracy stays small for all models.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE